# Deep Learning Concepts for Beginners

This notebook explains deep learning in **simple English**, step by step.
Every idea has a short explanation and a small code example.

**What you will learn:**
1. What is deep learning?
2. A single neuron
3. Activation functions
4. Layers and the forward pass
5. Loss (how wrong is the model?)
6. Gradient descent (how the model learns)
7. Backpropagation: train a small network
8. Overfitting
9. Summary

You only need `numpy` and `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)
print("Ready!")

## 1. What is deep learning?

- **Machine learning** = the computer learns patterns from data.
- **Deep learning** = machine learning that uses **neural networks** with many layers.
- "Deep" means many layers.

**Example:** A retail shop wants to guess if a customer will buy again.
The network reads data (age, visits, money spent) and gives an answer (a number between 0 and 1).

## 2. A single neuron

A neuron is the smallest part of a neural network. It does 3 things:

1. Multiply each input by a **weight**
2. Add everything together, then add a **bias**
3. Pass the result through an **activation function**

Formula: `output = activation(w1*x1 + w2*x2 + ... + bias)`

- **Weight** = how important an input is
- **Bias** = a small shift that helps the neuron fit better

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Two inputs: visits per month and money spent (scaled 0 to 1)
x = np.array([0.5, 0.8])

# Weights and bias (the model learns these)
w = np.array([0.4, 0.7])
b = -0.3

z = np.dot(x, w) + b       # weighted sum
out = sigmoid(z)           # activation

print("Weighted sum (z):", round(z, 3))
print("Neuron output   :", round(out, 3))

## 3. Activation functions

Without an activation function, a network can only learn **straight-line** patterns.
Activation functions add a **curve**, so the network can learn harder patterns.

Common ones:
- **Sigmoid**: output between 0 and 1 (good for yes/no answers)
- **ReLU**: output is 0 if the input is negative, otherwise the same number (most used in hidden layers)
- **Tanh**: output between -1 and 1

In [ ]:
def relu(z):
    return np.maximum(0, z)

z = np.linspace(-6, 6, 200)

plt.figure(figsize=(12, 3.5))
for i, (name, f) in enumerate([("Sigmoid", sigmoid), ("ReLU", relu), ("Tanh", np.tanh)]):
    plt.subplot(1, 3, i + 1)
    plt.plot(z, f(z))
    plt.title(name)
    plt.grid(True)
plt.tight_layout()
plt.show()

## 4. Layers and the forward pass

A network has layers:
- **Input layer**: your data goes in
- **Hidden layers**: the network finds patterns
- **Output layer**: the final answer

**Forward pass** = data moves from input to output, layer by layer.

In [ ]:
# A network with 3 inputs -> 4 hidden neurons -> 1 output
X = np.array([[0.2, 0.5, 0.9]])      # one customer (1 row, 3 features)

W1 = np.random.randn(3, 4)           # weights: input -> hidden
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1)           # weights: hidden -> output
b2 = np.zeros((1, 1))

hidden = relu(X @ W1 + b1)           # hidden layer
output = sigmoid(hidden @ W2 + b2)   # output layer

print("Hidden layer values:", np.round(hidden, 3))
print("Final prediction   :", np.round(output, 3))
print("(The weights are random now, so the answer is not smart yet.)")

## 5. Loss: how wrong is the model?

The **loss** is one number that tells us how far the prediction is from the real answer.
- Small loss = good
- Big loss = bad

A simple loss is **Mean Squared Error (MSE)**:

`loss = average of (prediction - actual)^2`

The goal of training: **make the loss as small as possible.**

In [ ]:
def mse(pred, actual):
    return np.mean((pred - actual) ** 2)

actual = np.array([1, 0, 1, 1])

good_pred = np.array([0.9, 0.1, 0.8, 0.9])
bad_pred  = np.array([0.2, 0.9, 0.3, 0.1])

print("Loss of good predictions:", round(mse(good_pred, actual), 3))
print("Loss of bad predictions :", round(mse(bad_pred, actual), 3))

## 6. Gradient descent: how the model learns

Imagine you are on a hill in the fog and want to reach the bottom.
You feel the slope under your feet and take a small step **downhill**. You repeat this again and again.

- The hill = the loss
- The slope = the **gradient**
- The step size = the **learning rate**

Update rule: `new_weight = old_weight - learning_rate * gradient`

Below, we find the lowest point of `loss = (w - 3)^2`. The best `w` is 3.

In [ ]:
w = 10.0              # start far away
learning_rate = 0.1
history = [w]

for step in range(30):
    gradient = 2 * (w - 3)          # slope of (w-3)^2
    w = w - learning_rate * gradient
    history.append(w)

print("Final w:", round(w, 4), "(best value is 3)")

plt.plot(history, marker="o")
plt.axhline(3, color="red", linestyle="--", label="best w = 3")
plt.xlabel("Step")
plt.ylabel("w")
plt.title("Gradient descent moving to the best value")
plt.legend()
plt.show()

**Learning rate tip:**
- Too small = learning is very slow
- Too big = the model jumps around and never settles
- A good value is in between

## 7. Backpropagation: train a small network

**Backpropagation** is how a network finds the gradient for **every** weight.
It works backward: from the loss, back through each layer.

Full training loop (repeat many times):
1. **Forward pass**: make a prediction
2. **Loss**: measure the error
3. **Backward pass**: find gradients
4. **Update**: change weights with gradient descent

We will train a tiny network on the **XOR** problem.
A single neuron cannot solve XOR, but a network with a hidden layer can.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

np.random.seed(0)
W1 = np.random.randn(2, 4)
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1)
b2 = np.zeros((1, 1))

learning_rate = 1.0
losses = []

for epoch in range(10000):
    # 1. Forward pass
    h = sigmoid(X @ W1 + b1)
    pred = sigmoid(h @ W2 + b2)

    # 2. Loss
    loss = np.mean((pred - y) ** 2)
    losses.append(loss)

    # 3. Backward pass (chain rule)
    d_pred = 2 * (pred - y) / len(X) * pred * (1 - pred)
    dW2 = h.T @ d_pred
    db2 = d_pred.sum(axis=0, keepdims=True)

    d_h = d_pred @ W2.T * h * (1 - h)
    dW1 = X.T @ d_h
    db1 = d_h.sum(axis=0, keepdims=True)

    # 4. Update weights
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

print("Final loss:", round(losses[-1], 5))
print("Predictions:", np.round(pred.ravel(), 2))
print("Expected   :", y.ravel())

plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss goes down while the network learns")
plt.show()

You should see the loss fall and the predictions get close to `[0, 1, 1, 0]`.
This is the heart of deep learning.

> Real projects use libraries like **TensorFlow** or **PyTorch**. They do the backward pass for you automatically.

## 8. Overfitting

**Overfitting** = the model remembers the training data instead of learning the real pattern.
It scores very well on training data but badly on new data.

**How to spot it:** training loss keeps going down, but validation (test) loss goes up.

**How to reduce it:**
- Get more data
- Use a smaller model
- Use **dropout** (randomly turn off some neurons while training)
- **Early stopping** (stop training when validation loss starts rising)

In [ ]:
# A picture of overfitting (example numbers)
epochs = np.arange(1, 21)
train_loss = 1 / epochs
val_loss = 1 / epochs[:8].max() + 0.02 * (epochs - 8).clip(min=0) + 0.1

plt.plot(epochs, train_loss, label="Training loss")
plt.plot(epochs, np.where(epochs <= 8, 1/epochs + 0.1, val_loss), label="Validation loss")
plt.axvline(8, color="gray", linestyle="--", label="Stop here")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Overfitting after epoch 8")
plt.show()

## 9. Summary

| Concept | Simple meaning |
|---|---|
| Neuron | Takes inputs, applies weights, gives one output |
| Weight / Bias | Numbers the model learns |
| Activation | Adds a curve so the model can learn hard patterns |
| Layer | A group of neurons |
| Forward pass | Data flows input to output |
| Loss | How wrong the prediction is |
| Gradient descent | Small steps to reduce the loss |
| Backpropagation | Finds the gradient for each weight |
| Epoch | One full pass over the training data |
| Overfitting | Model remembers data instead of learning |

### Next steps
- Try changing `learning_rate` in section 7 and see what happens
- Add more hidden neurons
- Learn **PyTorch** or **TensorFlow/Keras**
- Try a real dataset (for example, customer data from Kaggle)